# 1.0 Data Cleaning Pipeline

Following the structured research-backed data cleaning workflow:
- Step 1: Structure (Unit of analysis, column naming, type consistency, dropping dead columns)
- Step 2: Validity Checks (Categories, email/phone format sanity, date ranges)
- Step 3: Duplicates (Exact row duplicates, ID collisions, near-duplicate contact detection)
- Step 4: Missing Data (Name resolution, null imputation rules, missing mechanisms)
- Step 5: Outliers and Anomalies (Date plausibility, irregular phone lengths, abnormal notes)

## Setup and Load Raw Data (Untouched)

In [1]:
import re
from pathlib import Path
import pandas as pd
import numpy as np

DATA_RAW = Path('../data/raw')
DATA_INTERIM = Path('../data/interim')
DATA_INTERIM.mkdir(parents=True, exist_ok=True)

df_raw = pd.read_csv(DATA_RAW / 'leads_seed.csv')
print(f'Loaded raw dataset: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns.')

Loaded raw dataset: 2049 rows, 22 columns.


## Step 1: Structure
- Confirm 1 row = 1 lead record
- Standardize column names to snake_case
- Prune dead HubSpot export columns with near-100% missing values
- Coerce baseline types

In [11]:
# Inspect raw columns and dtypes
print(df_raw.dtypes)

# Inspect dead or nearly empty columns
empty_cols = [col for col in df_raw.columns if df_raw[col].isnull().sum() / len(df_raw) > 0.95]
print('Columns with >95% missing values:', empty_cols)

# Standardize column names
rename_map = {
    'Record ID': 'record_id',
    'First Name': 'first_name',
    'Last Name': 'last_name',
    'Full Name': 'full_name',
    'Job Title': 'job_title',
    'Company Name': 'company_name',
    'Email': 'email',
    'Phone Number': 'phone_number',
    'Country/Region': 'country',
    'City': 'city',
    'Lead Status': 'lead_status',
    'Lifecycle Stage': 'lifecycle_stage',
    'Original Source': 'original_source',
    'Contact Owner': 'contact_owner',
    'Create Date': 'create_date',
    'Last Modified Date': 'last_modified_date',
    'Notes': 'notes'
}

keep_cols = [c for c in rename_map.keys() if c in df_raw.columns]
df = df_raw[keep_cols].rename(columns=rename_map).copy()
print(f'Structured dataset columns: {list(df.columns)}')

Record ID                         int64
First Name                       object
Last Name                        object
Full Name                        object
Job Title                        object
Company Name                     object
Email                            object
Phone Number                     object
Country/Region                   object
City                            float64
Lead Status                      object
Lifecycle Stage                  object
Original Source                  object
Original Source Drill-Down 1    float64
Contact Owner                    object
Create Date                      object
Last Modified Date               object
Notes                            object
Annual Revenue                  float64
Marketing contact status        float64
GDPR consent                    float64
Lead Score                      float64
dtype: object
Columns with >95% missing values: ['City', 'Original Source Drill-Down 1', 'Annual Revenue', 'Marketing co

## Step 2: Validity Checks
- Clean and normalize categorical labels (e.g. Lead Status whitespace/casing)
- Validate email strings (lowercase, strip whitespace, check @ symbol)
- Normalize and validate phone numbers into pure digit strings
- Parse dates and check logical ordering (created <= modified)

In [3]:
# 1. Lead Status normalization
def clean_status(val):
    if pd.isna(val):
        return 'Unknown'
    cleaned = str(val).strip().title()
    status_map = {
        'New': 'New',
        'Contacted': 'Contacted',
        'Connected': 'Connected',
        'Qualified': 'Qualified',
        'Opportunity': 'Opportunity',
        'Closed Won': 'Closed Won',
        'Closed Lost': 'Closed Lost'
    }
    return status_map.get(cleaned, cleaned)

df['lead_status'] = df['lead_status'].apply(clean_status)
print('Valid lead statuses:\n', df['lead_status'].value_counts())

# 2. Email validation
df['email'] = df['email'].fillna('').astype(str).str.strip().str.lower()
df['has_valid_email'] = df['email'].apply(lambda x: bool(re.match(r'^[^@]+@[^@]+\.[^@]+$', x)))
print('Valid email count:', df['has_valid_email'].sum(), 'of', len(df))

# 3. Phone validation & digits extraction
df['phone_digits'] = df['phone_number'].fillna('').astype(str).apply(lambda x: re.sub(r'\D', '', x))
df['phone_number'] = df['phone_number'].fillna('').astype(str).str.strip()

# 4. Dates parsing and logical check
df['created_at'] = pd.to_datetime(df['create_date'], errors='coerce')
df['updated_at'] = pd.to_datetime(df['last_modified_date'], errors='coerce')
date_inversions = (df['updated_at'].notna() & (df['updated_at'] < df['created_at'])).sum()
print(f'Records with updated_at earlier than created_at: {date_inversions}')

Valid lead statuses:
 lead_status
Qualified      321
Closed Lost    316
Connected      293
Opportunity    291
Contacted      282
Closed Won     276
New            270
Name: count, dtype: int64
Valid email count: 2049 of 2049
Records with updated_at earlier than created_at: 0


## Step 3: Duplicates
- Check for exact duplicate rows
- Check for duplicate Record IDs
- Surface near-duplicate clusters based on normalized phone digits and email

In [4]:
exact_dups = df.duplicated().sum()
id_dups = df['record_id'].duplicated().sum()
print(f'Exact full duplicates: {exact_dups}')
print(f'Duplicate Record IDs: {id_dups}')

# Check duplicates by contact points
email_counts = df[df['email'] != '']['email'].value_counts()
dup_emails = email_counts[email_counts > 1]
print(f'Emails appearing multiple times: {len(dup_emails)}')

phone_counts = df[df['phone_digits'] != '']['phone_digits'].value_counts()
dup_phones = phone_counts[phone_counts > 1]
print(f'Phone digit fingerprints appearing multiple times: {len(dup_phones)}')

# Sample candidate duplicate pairs
if len(dup_phones) > 0:
    sample_phone = dup_phones.index[0]
    print(f'\nSample duplicate group on phone {sample_phone}:')
    display(df[df['phone_digits'] == sample_phone][['record_id', 'first_name', 'last_name', 'email', 'company_name']])

Exact full duplicates: 0
Duplicate Record IDs: 0
Emails appearing multiple times: 58
Phone digit fingerprints appearing multiple times: 232

Sample duplicate group on phone 8801687679766:


,record_id,first_name,last_name,email,company_name
1981,100236792,Mateo,Ang,m.ang@acme.com.au,Acme Analytics
1982,100236793,NaN,NaN,mateoang@acme.com.au,Acme Robotics
1983,100236794,NaN,NaN,mateoa@acme.com.au,Acme Trading


## Step 4: Missing Data
- Address missing names (combining first/last vs full_name)
- Handle missing categorical and contact attributes

In [5]:
def resolve_lead_names(row):
    first = str(row['first_name']).strip() if pd.notna(row['first_name']) else ''
    last = str(row['last_name']).strip() if pd.notna(row['last_name']) else ''
    full = str(row['full_name']).strip() if pd.notna(row['full_name']) else ''
    
    if first or last:
        combined = f'{first} {last}'.strip()
        return first, last, combined
    elif full:
        parts = full.split(' ', 1)
        f = parts[0]
        l = parts[1] if len(parts) > 1 else ''
        return f, l, full
    return '', '', ''

resolved = df.apply(resolve_lead_names, axis=1)
df['first_name'] = [r[0] for r in resolved]
df['last_name'] = [r[1] for r in resolved]
df['full_name'] = [r[2] for r in resolved]

# Fill defaults for missing metadata
df['contact_owner'] = df['contact_owner'].fillna('Unassigned').astype(str).str.strip()
df['country'] = df['country'].fillna('Unknown').astype(str).str.strip()
df['city'] = df['city'].fillna('').astype(str).str.strip()
df['company_name'] = df['company_name'].fillna('').astype(str).str.strip()
df['notes'] = df['notes'].fillna('').astype(str).str.strip()

print('Missing full names remaining:', (df['full_name'] == '').sum())

Missing full names remaining: 0


## Step 5: Outliers and Anomalies
- Check phone digit length anomalies
- Check date range anomalies (unrealistic past or future dates)
- Flag abnormal notes lengths

In [6]:
# Phone digit length distributions
df['phone_len'] = df['phone_digits'].apply(lambda x: len(x) if x else 0)
print('Phone length value counts:\n', df[df['phone_len'] > 0]['phone_len'].value_counts())

# Suspicious short phones
short_phones = df[(df['phone_len'] > 0) & (df['phone_len'] < 7)]
print(f'Suspiciously short phone numbers (<7 digits): {len(short_phones)}')

# Date range check
print('Create date range:', df['created_at'].min(), 'to', df['created_at'].max())
future_leads = (df['created_at'] > pd.Timestamp.now()).sum()
print(f'Future create dates flagged: {future_leads}')

# Notes character length
df['notes_len'] = df['notes'].str.len()
print('Notes length summary:')
print(df['notes_len'].describe())

Phone length value counts:
 phone_len
12    842
11    746
13    323
10    138
Name: count, dtype: int64
Suspiciously short phone numbers (<7 digits): 0
Create date range: 2025-09-02 00:00:00 to 2026-06-19 00:00:00
Future create dates flagged: 0
Notes length summary:
count    2049.000000
mean       74.271840
std        22.430531
min        33.000000
25%        58.000000
50%        71.000000
75%        89.000000
max       160.000000
Name: notes_len, dtype: float64


## Save Cleaned Interim Dataset

In [7]:
export_columns = [
    'record_id', 'first_name', 'last_name', 'full_name',
    'job_title', 'company_name', 'email', 'phone_number', 'phone_digits',
    'country', 'city', 'lead_status', 'lifecycle_stage', 'original_source',
    'contact_owner', 'created_at', 'updated_at', 'notes'
]

df_clean = df[export_columns].copy()
out_path = DATA_INTERIM / 'leads_cleaned.csv'
df_clean.to_csv(out_path, index=False)
print(f'Cleaned dataset saved successfully to {out_path} with {len(df_clean)} records.')

Cleaned dataset saved successfully to ..\data\interim\leads_cleaned.csv with 2049 records.
